In [95]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold, cross_validate, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor, HistGradientBoostingRegressor
from sklearn.metrics import r2_score
import joblib

In [98]:
df = pd.read_csv("StudentPerformanceFactors.csv")
df.head()


,Hours_Studied,Attendance,Parental_Involvement,Access_to_Resources,Extracurricular_Activities,Sleep_Hours,Previous_Scores,Motivation_Level,Internet_Access,Tutoring_Sessions,Family_Income,Teacher_Quality,School_Type,Peer_Influence,Physical_Activity,Learning_Disabilities,Parental_Education_Level,Distance_from_Home,Gender,Exam_Score
0,23,84,Low,High,No,7,73,Low,Yes,0,Low,Medium,Public,Positive,3,No,High School,Near,Male,67
1,19,64,Low,Medium,No,8,59,Low,Yes,2,Medium,Medium,Public,Negative,4,No,College,Moderate,Female,61
2,24,98,Medium,Medium,Yes,7,91,Medium,Yes,2,Medium,Medium,Public,Neutral,4,No,Postgraduate,Near,Male,74
3,29,89,Low,Medium,Yes,8,98,Medium,Yes,1,Medium,Medium,Public,Negative,4,No,High School,Moderate,Male,71
4,19,92,Medium,Medium,Yes,6,65,Medium,Yes,3,Medium,High,Public,Neutral,4,No,College,Near,Female,70


In [56]:
df.shape

(6607, 20)

In [57]:
df.isnull().sum()

Hours_Studied                  0
Attendance                     0
Parental_Involvement           0
Access_to_Resources            0
Extracurricular_Activities     0
Sleep_Hours                    0
Previous_Scores                0
Motivation_Level               0
Internet_Access                0
Tutoring_Sessions              0
Family_Income                  0
Teacher_Quality               78
School_Type                    0
Peer_Influence                 0
Physical_Activity              0
Learning_Disabilities          0
Parental_Education_Level      90
Distance_from_Home            67
Gender                         0
Exam_Score                     0
dtype: int64

In [58]:
df.duplicated().sum()

np.int64(0)

In [59]:
df.describe()

,Hours_Studied,Attendance,Sleep_Hours,Previous_Scores,Tutoring_Sessions,Physical_Activity,Exam_Score
count,6607.000000,6607.000000,6607.00000,6607.000000,6607.000000,6607.000000,6607.000000
mean,19.975329,79.977448,7.02906,75.070531,1.493719,2.967610,67.235659
std,5.990594,11.547475,1.46812,14.399784,1.230570,1.031231,3.890456
min,1.000000,60.000000,4.00000,50.000000,0.000000,0.000000,55.000000
25%,16.000000,70.000000,6.00000,63.000000,1.000000,2.000000,65.000000
50%,20.000000,80.000000,7.00000,75.000000,1.000000,3.000000,67.000000
75%,24.000000,90.000000,8.00000,88.000000,2.000000,4.000000,69.000000
max,44.000000,100.000000,10.00000,100.000000,8.000000,6.000000,101.000000


In [60]:
df.dtypes

Hours_Studied                  int64
Attendance                     int64
Parental_Involvement          object
Access_to_Resources           object
Extracurricular_Activities    object
Sleep_Hours                    int64
Previous_Scores                int64
Motivation_Level              object
Internet_Access               object
Tutoring_Sessions              int64
Family_Income                 object
Teacher_Quality               object
School_Type                   object
Peer_Influence                object
Physical_Activity              int64
Learning_Disabilities         object
Parental_Education_Level      object
Distance_from_Home            object
Gender                        object
Exam_Score                     int64
dtype: object

In [61]:
X = df.drop('Exam_Score', axis=1)
y = df['Exam_Score']

In [62]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [63]:
numerical_features = X.select_dtypes(include=['number']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

In [64]:
numerical_features

['Hours_Studied',
 'Attendance',
 'Sleep_Hours',
 'Previous_Scores',
 'Tutoring_Sessions',
 'Physical_Activity']

In [65]:
categorical_features

['Parental_Involvement',
 'Access_to_Resources',
 'Extracurricular_Activities',
 'Motivation_Level',
 'Internet_Access',
 'Family_Income',
 'Teacher_Quality',
 'School_Type',
 'Peer_Influence',
 'Learning_Disabilities',
 'Parental_Education_Level',
 'Distance_from_Home',
 'Gender']

In [66]:
numerical_transformer = Pipeline(steps=[
    ("imputer" , SimpleImputer(strategy = "median")),
    ("scaler" , StandardScaler())
])
    
categorical_transformer = Pipeline(steps=[
    ("imputer" , SimpleImputer(strategy = "most_frequent")),
    ("onehot" , OneHotEncoder(handle_unknown='ignore'))
])

preprocess = ColumnTransformer(
    transformers=[
    ("num", numerical_transformer, numerical_features),
    ("cat", categorical_transformer, categorical_features)
    ]
)

In [67]:
models = {
    "Linear Regression" : LinearRegression(),
    "SVR" : SVR(),
    "Random Forest" : RandomForestRegressor(),
    "Gradient Boosting" : GradientBoostingRegressor(),
    "Ridge" : Ridge(),
    "Lasso" : Lasso(),
    "Ada Boost" : AdaBoostRegressor(),
    "Hist Gradient Boost" : HistGradientBoostingRegressor()
}

In [68]:
scoring = {
    "r2" : "r2",
    "mae" : "neg_mean_absolute_error",
    "rmse" : "neg_root_mean_squared_error"
}

In [69]:
k = 5
cv = KFold(n_splits = k, shuffle=True, random_state=42)


In [70]:
rows = []

for name, model in models.items():
    baseline = Pipeline(steps=[
    ("preprocess" , preprocess),
    ("model", model)
    ])
    scores = cross_validate(baseline, X_train, y_train, cv=cv, scoring = scoring , n_jobs=-1)
    rows.append({
        "name" : name,
        'r2' : scores["test_r2"].mean(),
        "mae" : -scores["test_mae"].mean(),
        "rmse" : -scores["test_rmse"].mean()
    })
cv_results = pd.DataFrame(rows).sort_values("r2",ascending= False)
print(cv_results)


                  name        r2       mae      rmse
4                Ridge  0.715588  0.508176  2.089528
0    Linear Regression  0.715585  0.508200  2.089537
1                  SVR  0.705634  0.555552  2.125861
3    Gradient Boosting  0.676149  0.859993  2.230308
7  Hist Gradient Boost  0.672550  0.875072  2.243205
2        Random Forest  0.615016  1.189245  2.432018
5                Lasso  0.399720  1.960651  3.037792
6            Ada Boost -0.882194  4.464340  5.336941


In [71]:
param_ridge = {
    "model__alpha" : [ 0.0001, 0.01, 0.1, 1, 10, 100],
    "model__solver": ["auto", "svd", "cholesky", "lsqr", "sag"]
}
pipe = Pipeline([
    ("preprocess", preprocess),
    ("model", Ridge())
])
ridge_grid = GridSearchCV(
    pipe,
    param_ridge,
    cv=cv,
    scoring="r2",
    n_jobs=-1,
    verbose=1
)
ridge_grid.fit(X_train, y_train)
print(ridge_grid.best_score_)
print(ridge_grid.best_params_)


Fitting 5 folds for each of 30 candidates, totalling 150 fits
0.7155948388028647
{'model__alpha': 10, 'model__solver': 'sag'}


In [76]:
param_svr = {
    "model__C": [0.1, 0.5, 10, 100],
    "model__gamma": ["scale", 0.01, 0.1],
    "model__kernel": ["rbf"]
}
pipe = Pipeline([
    ("preprocess", preprocess),
    ("model", SVR())
])
svr_grid = GridSearchCV(
    pipe,
    param_svr,
    cv=cv,
    scoring="r2",
    n_jobs=-1,
    verbose=1
)
svr_grid.fit(X_train, y_train)
print(svr_grid.best_score_)


Fitting 5 folds for each of 12 candidates, totalling 60 fits
0.7145078254763033


In [75]:
print(svr_grid.best_params_)

{'model__C': 1, 'model__gamma': 0.01, 'model__kernel': 'rbf'}


In [ ]:
param_hist = {
    "model__learning_rate": [0.01, 0.05, 0.1],
    "model__max_depth": [None, 10],
    "model__max_iter": [100, 200, 300]
}
pipe = Pipeline([
    ("preprocess", preprocess),
    ("model", HistGradientBoostingRegressor())
    
])
hist_grid = GridSearchCV(
    pipe,
    param_hist,
    cv=cv,
    scoring="r2",
    n_jobs=-1,
    verbose=1
)
hist_grid.fit(X_train, y_train)
print(hist_grid.best_score_)
print(hist_grid.best_params_)

Fitting 5 folds for each of 18 candidates, totalling 90 fits
0.6769821405720793
{'model__learning_rate': 0.05, 'model__max_depth': None, 'model__max_iter': 200}


In [ ]:
final_base = Pipeline(
        steps=[
        ("preprocess", preprocess),
        ("model", Ridge(
            alpha=10,
            solver='sag'
        ))
        ]
)

In [94]:
final_base.fit(X_train,y_train)
y_train_pred = final_base.predict(X_train)
r2_train = r2_score(y_train,y_train_pred)
r2_train

0.7173137260678102

In [91]:
final_base.fit(X_train,y_train)
y_pred = final_base.predict(X_test)
r2 = r2_score(y_test,y_pred)
r2

0.7696584630608467

In [97]:
joblib.dump(final_base,"performance_model.pkl")

['performance_model.pkl']